# Qwen2.5-Coder-7B QLoRA Fine-tuning v2 (Unsloth) — OpenCodeInstruct (chọn theo keyword, ChatML)

Bản v2 sau khi phân tích kết quả của `fine-tune-qwen-unsloth.ipynb` (3 epoch: HumanEval 78.7 → 77.4 → 76.0, giảm dần qua các epoch, không hơn base có ý nghĩa thống kê).

**Thay đổi so với v1:**
1. **Loss chỉ tính trên phần assistant** (`train_on_responses_only`) — v1 học cả phần đề bài của user.
2. **Học lại embedding của token ChatML** (`embed_tokens` + `lm_head`) khi base là model *base* — `<|im_start|>/<|im_end|>` chưa được train kỹ trong Qwen base, smoke test v1 sinh rác sau code block thay vì dừng ở `<|im_end|>`. Bỏ qua nếu fine-tune từ `-Instruct` (đã biết ChatML).
3. **1 epoch, lr 1e-4, r=32 + rsLoRA** (v1: 3 epoch, 2e-4, r=16; điểm tốt nhất ở epoch 1 rồi giảm).
4. **Tách 3% validation** + eval loss định kỳ + `load_best_model_at_end` để chọn checkpoint theo eval loss thay vì đoán.
5. Smoke test in output **không bỏ special token** và kiểm tra model có tự dừng ở `<|im_end|>` không.

**Chưa được chạy thử** — các tham số (nhất là `TRAIN_EMBEDS` trên T4 15GB) cần theo dõi VRAM ở lần chạy đầu.

**Dataset:** `chatml_v4.jsonl` — cột `text`, định dạng Qwen ChatML, tạo bằng `src/dataset/count_related_problems.py --hq --max-tokens 1024 --max-per-keyword N --system-prompt --out chatml_v4.jsonl`. Nên xuất kèm `--system-prompt`: chat template của Qwen tự thêm system prompt mặc định khi eval, nên dữ liệu train cũng phải có để khớp.

In [ ]:
!pip install -q -U unsloth unsloth_zoo bitsandbytes
import importlib.metadata as md
for p in ["unsloth", "unsloth_zoo", "bitsandbytes", "trl", "peft", "transformers", "accelerate", "torch"]:
    print(f"{p:14s}", md.version(p))

## 1. Imports, Config & Login

In [ ]:
import os
import glob
import json
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

REPORT_TO = "none"

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"
HUB_MODEL_NAME = "hoangnh39/qwen2.5-coder-7b-opencode-v4"

NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
LORA_R = 32
LORA_ALPHA = 32       
SEED = 3407

# Train embed_tokens + lm_head để học <|im_start|>/<|im_end|> (Qwen base chưa train kỹ hai token này).
# Tốn thêm VRAM (full weight ~2 x 545M tham số): nếu OOM trên T4 thì đặt False và chỉ dựa vào dừng theo chuỗi khi eval.
TRAIN_EMBEDS = True
EMBED_LR = 1e-5           # Unsloth khuyến nghị thấp hơn lr của LoRA 2-10 lần

VAL_FRACTION = 0.03
EVAL_STEPS = 200
MAX_VAL_SAMPLES = 300      # eval ít mẫu hơn để mỗi lần eval không tốn nhiều thời gian

HF_DATASET_ID = None
DATASET_FILENAME = "chatml_v4.jsonl"

MAX_SEQ_LENGTH = 1024
MAX_TRAIN_SAMPLES = None

## 2. Load Base Model + Attach LoRA (Unsloth)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect bf16/fp16 support
    load_in_4bit=True,
)

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# embed_tokens/lm_head train đầy đủ (không phải LoRA) qua modules_to_save: ~+4GB VRAM, theo dõi "GPU mem" bên dưới
modules_to_save = ["embed_tokens", "lm_head"] if TRAIN_EMBEDS else None

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,                       # 0 is required for Unsloth's fused kernels
    target_modules=target_modules,
    modules_to_save=modules_to_save,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=True,
    random_state=SEED,
)
model.print_trainable_parameters()
print("GPU mem:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. Load Dataset (OpenCodeInstruct v4, ChatML) + Train/Val split

In [ ]:
if HF_DATASET_ID:
    dataset = load_dataset(HF_DATASET_ID, split="train")
else:
    matches = glob.glob(f"/kaggle/input/**/{DATASET_FILENAME}", recursive=True)
    assert matches, f"Không thấy {DATASET_FILENAME} trong /kaggle/input — hãy attach Kaggle Dataset chứa file này."
    print("Using", matches[0])
    dataset = load_dataset("json", data_files=matches[0], split="train")

assert "text" in dataset.column_names, f"Cần cột 'text', hiện có: {dataset.column_names}"
print(f"Loaded {len(dataset):,} examples")

def n_tokens(batch):
    return {"n_tok": [len(ids) for ids in tokenizer(batch["text"], add_special_tokens=False)["input_ids"]]}

dataset = dataset.map(n_tokens, batched=True, batch_size=1000)
before = len(dataset)
dataset = dataset.filter(lambda ex: ex["n_tok"] <= MAX_SEQ_LENGTH).remove_columns("n_tok")
print(f"Kept {len(dataset):,}/{before:,} samples with <= {MAX_SEQ_LENGTH} tokens")

dataset = dataset.shuffle(seed=SEED)
if MAX_TRAIN_SAMPLES:
    dataset = dataset.select(range(min(MAX_TRAIN_SAMPLES, len(dataset))))
split = dataset.train_test_split(test_size=VAL_FRACTION, seed=SEED)
train_ds = split["train"]
val_ds = split["test"].select(range(min(MAX_VAL_SAMPLES, len(split["test"]))))
print(f"Train {len(train_ds):,} | Val {len(val_ds):,}")
print(train_ds[0]["text"][:500])
if "<|im_start|>system" not in train_ds[0]["text"]:
    print("WARNING: dữ liệu không có system prompt, nhưng chat template của Qwen tự thêm khi eval -> nên xuất lại với --system-prompt")

## 4. Train (loss chỉ trên phần assistant)

In [ ]:
from huggingface_hub import HfApi
from transformers import TrainerCallback
from unsloth import UnslothTrainer, UnslothTrainingArguments
from unsloth.chat_templates import train_on_responses_only

class PushEpochAdapter(TrainerCallback):
    """Sau mỗi epoch: lưu LoRA adapter và đẩy lên branch `epoch-N` của HUB_MODEL_NAME."""
    def __init__(self, tokenizer):
        self.api, self.tokenizer = HfApi(), tokenizer
        self.api.create_repo(HUB_MODEL_NAME, private=True, exist_ok=True)

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch))
        branch, path = f"epoch-{epoch}", f"./adapter_epoch{epoch}"
        model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        self.api.create_branch(HUB_MODEL_NAME, branch=branch, exist_ok=True)
        self.api.upload_folder(repo_id=HUB_MODEL_NAME, folder_path=path, revision=branch,
                               commit_message=f"LoRA adapter after epoch {epoch}")
        print(f"Pushed epoch {epoch} adapter -> {HUB_MODEL_NAME}@{branch}")

train_args = UnslothTrainingArguments(
    output_dir="./checkpoints",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch size 16
    optim="adamw_8bit",
    learning_rate=LEARNING_RATE,
    embedding_learning_rate=EMBED_LR if TRAIN_EMBEDS else None,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
    report_to=REPORT_TO,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=train_args,
    callbacks=[PushEpochAdapter(tokenizer)],
)

# Mask phần user: loss chỉ tính từ sau "<|im_start|>assistant\n"
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Kiểm tra mask: phần user phải là -100, phần assistant mới có label
ex = trainer.train_dataset[0]
print("Labels kept:", sum(l != -100 for l in ex["labels"]), "/", len(ex["labels"]))
print(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])[:300])

trainer_stats = trainer.train()
print(trainer_stats)

## 5. Save / Push Adapter (best checkpoint theo eval loss)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint, "| best eval_loss:", trainer.state.best_metric)
model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Final adapter pushed to https://huggingface.co/{HUB_MODEL_NAME}")

## 6. Smoke Test — model có tự dừng ở `<|im_end|>` không?

Không dùng `StopOnCodeBlockEnd` và in output **kèm special token**. v1 sinh rác (`โต๊ะ ... assistant`) sau code block ở bước này.

In [ ]:
FastLanguageModel.for_inference(model)

task = "Write a Python function `is_palindrome(s: str) -> bool` that checks whether a string is a palindrome, ignoring case and non-alphanumeric characters."
# Dùng đúng chat template như EvalPlus/vLLM sẽ dùng khi eval (có system prompt mặc định của Qwen)
prompt = tokenizer.apply_chat_template([{"role": "user", "content": task}], tokenize=False, add_generation_prompt=True)
print(prompt)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=400, temperature=0.2, do_sample=True,
                     eos_token_id=[im_end_id, tokenizer.eos_token_id], pad_token_id=tokenizer.eos_token_id)
gen = out[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(gen, skip_special_tokens=False))
print("\n---> stopped on <|im_end|>:", gen[-1].item() == im_end_id, "| new tokens:", len(gen))